## Подключение библиотек

In [7]:
# Основные библиотеки
import os
import cv2
import glob
import json
import numpy as np
from tqdm import tqdm
from collections import defaultdict, Counter

# YOLO
import supervision as sv
from ultralytics import YOLO

# Torch
import torch
import torchvision.transforms as transforms
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader

# Модели
from movinets import MoViNet
from movinets.config import _C

## Трекинг

In [ ]:
# Загрузка модели
model = YOLO('../models/detector/yolo/yolov8n.pt') # 'n' - nano (самая быстрая), можно 's', 'm', 'l' для большей точности
video_path = '../data/test_video.mp4'
cap = cv2.VideoCapture(video_path)

# Создаём папки
raw_dir = '../data/temp_raw_frames'  # Временная папка для сырых кадров
final_dir = '../data/final_tracks_bot_tracker_resnet50'  # Финальные видео
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(final_dir, exist_ok=True)

# Инициализация
tracker = sv.ByteTrack()

# Структуры для хранения данных
all_tracks = defaultdict(list)  # track_id -> список кортежей (кадр, координаты)
track_metadata = {}             # Метаданные треков
frame_count = 0

In [8]:
print("="*60)
print("ЭТАП 1: Сбор всех кадров для каждого трека")
print("="*60)

# Получаем общее количество кадров в видео
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# ПЕРВЫЙ ПРОХОД: собираем все кадры
with tqdm(total=total_frames, desc="Обработка видео", unit="кадр", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]') as pbar:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        results = model.track(frame, persist=True, classes=[0], verbose=False)
        
        if results[0].boxes.id is not None:
            detections = sv.Detections.from_ultralytics(results[0])
            detections = tracker.update_with_detections(detections)
            
            for i in range(len(detections)):
                bbox = detections.xyxy[i]
                track_id = detections.tracker_id[i]
                
                # Добавляем паддинг
                padding = 30
                x1, y1, x2, y2 = map(int, bbox)
                h, w = frame.shape[:2]
                
                x1_pad = max(0, x1 - padding)
                y1_pad = max(0, y1 - padding)
                x2_pad = min(w, x2 + padding)
                y2_pad = min(h, y2 + padding)
                
                # Вырезаем и сохраняем кадр во временную папку
                crop = frame[y1_pad:y2_pad, x1_pad:x2_pad]
                
                if crop.shape[0] > 50 and crop.shape[1] > 50:
                    # Сохраняем кадр как изображение
                    frame_filename = f'{raw_dir}/track_{track_id}_frame_{frame_count:04d}.jpg'
                    cv2.imwrite(frame_filename, crop)
                    
                    # Сохраняем метаданные
                    all_tracks[track_id].append({
                        'frame_num': frame_count,
                        'filename': frame_filename,
                        'coords': (x1_pad, y1_pad, x2_pad, y2_pad),
                        'size': (crop.shape[1], crop.shape[0])  # (width, height)
                    })
        
        # Обновляем прогресс-бар каждый кадр
        pbar.update(1)
        
        # Дополнительная информация в прогресс-баре
        if frame_count % 50 == 0:
            pbar.set_postfix({
                'треков': len(all_tracks),
                'активных': len([t for t in all_tracks.values() if t])
            })

cap.release()
print(f"\nСбор завершён. Всего кадров: {frame_count}")
print(f"Собрано треков: {len(all_tracks)}")

ЭТАП 1: Сбор всех кадров для каждого трека


Обработка видео: 100%|██████████| 7440/7440 [19:42<00:00]  


Сбор завершён. Всего кадров: 7440
Собрано треков: 15


In [9]:
# Анализ собранных данных
print("\n" + "="*60)
print("ЭТАП 2: Анализ и определение максимальных размеров")
print("="*60)

for track_id, frames in tqdm(all_tracks.items(), desc="Анализ кадров", unit="трек", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'):
    if not frames:
        continue
    
    # Находим максимальную ширину и высоту для этого трека
    widths = [f['size'][0] for f in frames]
    heights = [f['size'][1] for f in frames]
    
    max_width = max(widths)
    max_height = max(heights)
    
    track_metadata[track_id] = {
        'max_width': max_width,
        'max_height': max_height,
        'num_frames': len(frames),
        'frames': frames
    }


ЭТАП 2: Анализ и определение максимальных размеров


Анализ кадров: 100%|██████████| 15/15 [00:00<00:00]


In [10]:
# ВТОРОЙ ПРОХОД: создание финальных видео
print("\n" + "="*60)
print("ЭТАП 3: Создание финальных видео с постоянным размером")
print("="*60)

for track_id, meta in tqdm(track_metadata.items(), desc="Формирование видео", unit="трек", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'):
    if meta['num_frames'] < 10:  # Пропускаем очень короткие треки
        continue
    
    output_path = f'{final_dir}/person_{track_id}.avi'
    max_w, max_h = meta['max_width'], meta['max_height']
    
    # Делаем размер чётным (требование кодеков)
    if max_w % 2 != 0:
        max_w += 1
    if max_h % 2 != 0:
        max_h += 1
    
    # Создаём VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(output_path, fourcc, 20.0, (max_w, max_h))
    
    if not out.isOpened():
        continue
    
    frames_written = 0
    
    # Обрабатываем каждый кадр трека
    for frame_data in sorted(meta['frames'], key=lambda x: x['frame_num']):
        # Загружаем сохранённый кадр
        crop = cv2.imread(frame_data['filename'])
        
        if crop is not None:
            # Ресайзим к максимальному размеру с сохранением пропорций
            h, w = crop.shape[:2]
            
            # Если кадр меньше максимального размера - добавляем чёрные границы
            if w < max_w or h < max_h:
                # Создаём чёрный холст максимального размера
                canvas = np.zeros((max_h, max_w, 3), dtype=np.uint8)
                
                # Центрируем оригинальный кадр на холсте
                x_offset = (max_w - w) // 2
                y_offset = (max_h - h) // 2
                
                canvas[y_offset:y_offset+h, x_offset:x_offset+w] = crop
                final_frame = canvas
            else:
                # Если кадр больше или равен максимальному - просто ресайзим
                final_frame = cv2.resize(crop, (max_w, max_h))
            
            # Записываем кадр
            out.write(final_frame)
            frames_written += 1
    
    out.release()


ЭТАП 3: Создание финальных видео с постоянным размером


Формирование видео: 100%|██████████| 15/15 [05:56<00:00]


In [11]:
# Очистка временных файлов (опционально)
print("\n" + "="*60)
print("ЭТАП 4: Очистка временных файлов")
print("="*60)

# Удаляем временные JPEG файлы
temp_files = glob.glob(f'{raw_dir}/*.jpg')
for f in temp_files:
    os.remove(f)
os.rmdir(raw_dir)

print(f"Удалено временных файлов: {len(temp_files)}")
print(f"Временная папка {raw_dir} удалена")

# 8. Итоговая статистика
print("\n" + "="*60)
print("ИТОГОВЫЙ ОТЧЁТ")
print("="*60)

final_videos = glob.glob(f'{final_dir}/*.avi')
total_size_kb = sum(os.path.getsize(f) for f in final_videos) / 1024

print(f"Обработано кадров исходного видео: {frame_count}")
print(f"Обнаружено уникальных треков: {len(all_tracks)}")
print(f"Создано финальных видео: {len(final_videos)}")
print(f"Общий размер видео: {total_size_kb:.1f} КБ")
print(f"Финальные видео сохранены в: {final_dir}")

# Показываем информацию о каждом финальном видео
print("\nДетальная информация по видео:")
for video_path in final_videos:
    cap_test = cv2.VideoCapture(video_path)
    if cap_test.isOpened():
        frames = int(cap_test.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap_test.get(cv2.CAP_PROP_FPS)
        width = int(cap_test.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap_test.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap_test.release()
        
        filename = os.path.basename(video_path)
        size_kb = os.path.getsize(video_path) / 1024
        
        print(f"  {filename}: {width}x{height}, {frames} кадров, {size_kb:.1f} КБ")


ЭТАП 4: Очистка временных файлов
Удалено временных файлов: 22853
Временная папка ../data/temp_raw_frames удалена

ИТОГОВЫЙ ОТЧЁТ
Обработано кадров исходного видео: 7440
Обнаружено уникальных треков: 15
Создано финальных видео: 12
Общий размер видео: 750132.8 КБ
Финальные видео сохранены в: ../data/final_tracks

Детальная информация по видео:
  person_1.avi: 514x620, 7440 кадров, 263229.8 КБ
  person_2.avi: 524x604, 7440 кадров, 296664.9 КБ
  person_3.avi: 500x560, 7436 кадров, 185441.4 КБ
  person_4.avi: 154x428, 41 кадров, 409.9 КБ
  person_6.avi: 174x444, 150 кадров, 1213.7 КБ
  person_9.avi: 204x508, 57 кадров, 526.7 КБ
  person_10.avi: 236x500, 19 кадров, 264.6 КБ
  person_12.avi: 190x500, 16 кадров, 188.4 КБ
  person_14.avi: 200x502, 29 кадров, 311.7 КБ
  person_19.avi: 158x512, 32 кадров, 270.6 КБ
  person_20.avi: 230x502, 25 кадров, 308.1 КБ
  person_26.avi: 134x504, 155 кадров, 1303.1 КБ


## Распознавание

In [23]:
def get_transforms(mode='train', input_size=172):
    """Правильные трансформации для видео с учетом batch dimension"""
    
    def random_horizontal_flip_video(video):
        """Случайное горизонтальное отражение для видео [C, T, H, W] или [B, C, T, H, W]"""
        if video.ndim == 4:  # [C, T, H, W]
            if torch.rand(1) < 0.5:
                return video.flip(3)  # flip width dimension
            return video
        elif video.ndim == 5:  # [B, C, T, H, W]
            if torch.rand(1) < 0.5:
                return video.flip(4)  # flip width dimension
            return video
    
    def center_crop_video(video, size):
        """Центральный кроп для видео"""
        if video.ndim == 4:  # [C, T, H, W]
            _, _, H, W = video.shape
            top = (H - size[0]) // 2
            left = (W - size[1]) // 2
            return video[:, :, top:top+size[0], left:left+size[1]]
        elif video.ndim == 5:  # [B, C, T, H, W]
            _, _, _, H, W = video.shape
            top = (H - size[0]) // 2
            left = (W - size[1]) // 2
            return video[:, :, :, top:top+size[0], left:left+size[1]]
    
    # Упрощенная версия без resize_video - будем использовать только center_crop
    if mode == 'train':
        return transforms.Compose([
            transforms.Lambda(lambda x: x.float()),
            transforms.Lambda(lambda x: random_horizontal_flip_video(x)),
            transforms.Lambda(lambda x: center_crop_video(x, (input_size, input_size))),
        ])
    else:  # test/val
        return transforms.Compose([
            transforms.Lambda(lambda x: x.float()),
            transforms.Lambda(lambda x: center_crop_video(x, (input_size, input_size))),
        ])

In [24]:
def predict_video_actions_streaming(model, video_path, class_names, device='cuda'):
    """
    Стриминговая обработка видео - обрабатывает кадры по мере чтения
    """
    # Настройки
    SEGMENT_FRAMES = 16  # MoViNet обучена на 16 кадрах
    OVERLAP = 0.5  # 50% перекрытие
    INPUT_SIZE = 172  # MoViNet A0 ожидает 172x172
    TARGET_FPS = 30  # Целевой FPS
    TARGET_SIZE = (INPUT_SIZE, INPUT_SIZE)  # Целевой размер кадра
    
    print("="*60)
    print("СТРИМИНГОВЫЙ АНАЛИЗ ВИДЕО")
    print("="*60)
    
    # Открываем видео
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Ошибка: Не удалось открыть видео {video_path}")
        return []
    
    # Получаем информацию о видео
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    if original_fps <= 0:
        original_fps = TARGET_FPS
        print(f"Предупреждение: FPS не определен, используем {TARGET_FPS}")
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Видео: {video_path}")
    print(f"Размер: {width}x{height}, FPS: {original_fps:.2f}")
    print(f"Длительность: {total_frames/original_fps:.1f} сек, Кадров: {total_frames}")
    print(f"Целевой размер: {TARGET_SIZE[0]}x{TARGET_SIZE[1]}")
    
    # Вычисляем шаг для пропуска кадров (если FPS выше целевого)
    skip_ratio = max(1, original_fps / TARGET_FPS)
    print(f"Пропуск кадров: 1 из каждых {skip_ratio:.1f} кадров")
    
    # Вычисляем шаг для сегментов
    step = int(SEGMENT_FRAMES * (1 - OVERLAP))
    if step <= 0:
        step = 1
    
    print(f"\nПараметры сегментации:")
    print(f"Длина сегмента: {SEGMENT_FRAMES} кадров")
    print(f"Перекрытие: {OVERLAP*100:.0f}%")
    print(f"Шаг: {step} кадров")
    
    # Инициализация буферов
    frame_buffer = []  # Буфер для кадров
    results = []       # Результаты
    frame_counter = 0  # Счетчик всех кадров
    processed_frame_counter = 0  # Счетчик обработанных кадров
    segment_counter = 0  # Счетчик сегментов
    
    # Упрощенные трансформации
    transform = get_transforms(input_size=INPUT_SIZE)
    model.eval()
    
    print("\nНачало стриминг-обработки...")
    
    # Прогресс-бар
    from tqdm import tqdm
    pbar = tqdm(total=total_frames, desc="Обработка видео", unit="кадр")
    
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Пропускаем кадры для downsampling FPS
            if frame_counter % int(skip_ratio) == 0:
                # Конвертируем BGR -> RGB и уменьшаем размер
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame_resized = cv2.resize(frame_rgb, TARGET_SIZE)
                frame_buffer.append(frame_resized)
                processed_frame_counter += 1
                
                # Когда накопили достаточно кадров для сегмента И пришло время для нового сегмента
                if (len(frame_buffer) >= SEGMENT_FRAMES and 
                    (processed_frame_counter - SEGMENT_FRAMES) % step == 0):
                    
                    # Берем последние SEGMENT_FRAMES кадров
                    if len(frame_buffer) > SEGMENT_FRAMES:
                        segment_frames = frame_buffer[-SEGMENT_FRAMES:]
                    else:
                        segment_frames = frame_buffer
                    
                    # Подготавливаем сегмент
                    segment_np = np.array(segment_frames, dtype=np.float32)  # [T, H, W, 3]
                    segment_np = np.transpose(segment_np, (3, 0, 1, 2))  # [3, T, H, W]
                    segment_np = segment_np / 255.0  # Нормализация
                    
                    # Конвертируем в тензор
                    segment_tensor = torch.from_numpy(segment_np).unsqueeze(0)  # [1, 3, T, H, W]
                    
                    # Применяем трансформации
                    segment_tensor = transform(segment_tensor)
                    segment_tensor = segment_tensor.to(device)
                    
                    # Предсказание
                    outputs = model(segment_tensor)
                    probs = torch.softmax(outputs, dim=1)
                    confidence, predicted_class = torch.max(probs, dim=1)
                    
                    # Вычисляем временные метки
                    start_frame_idx = processed_frame_counter - SEGMENT_FRAMES
                    start_time = (start_frame_idx * skip_ratio) / original_fps
                    end_time = (processed_frame_counter * skip_ratio) / original_fps
                    
                    # Получаем название класса
                    pred_class_idx = predicted_class.item()
                    class_name = class_names.get(str(pred_class_idx), 
                                               f"Class_{pred_class_idx}")
                    
                    # Сохраняем результат
                    results.append({
                        'segment_id': segment_counter,
                        'start_time': max(0, start_time),
                        'end_time': end_time,
                        'predicted_class': pred_class_idx,
                        'class_name': class_name,
                        'confidence': confidence.item(),
                        'center_time': (start_time + end_time) / 2
                    })
                    
                    segment_counter += 1
            
            frame_counter += 1
            pbar.update(1)
    
    pbar.close()
    cap.release()
    
    print(f"\n✓ Обработка завершена!")
    print(f"  Всего кадров: {frame_counter}")
    print(f"  Обработано кадров: {processed_frame_counter}")
    print(f"  Создано сегментов: {segment_counter}")
    
    # Статистика
    if results:
        from collections import Counter
        
        print("\n" + "="*60)
        print("РЕЗУЛЬТАТЫ АНАЛИЗА")
        print("="*60)
        
        # Топ-5 наиболее частых действий
        pred_classes = [r['predicted_class'] for r in results]
        pred_counter = Counter(pred_classes)
        
        print(f"\nТоп-5 действий в видео:")
        for class_id, count in pred_counter.most_common(5):
            class_name = class_names.get(str(class_id), f"Class_{class_id}")
            percentage = (count / len(results)) * 100
            print(f"  {class_name}: {count} сегментов ({percentage:.1f}%)")
        
        # Действия с высокой уверенностью
        high_conf = [r for r in results if r['confidence'] > 0.7]
        if high_conf:
            print(f"\nДействия с высокой уверенностью (>70%): {len(high_conf)}")
            for r in high_conf[:3]:
                print(f"  {r['start_time']:.1f}-{r['end_time']:.1f}с: "
                      f"{r['class_name']} ({r['confidence']:.1%})")
        
        # Временная шкала (упрощенная)
        print(f"\nВременная шкала основных действий:")
        if len(results) > 0:
            # Группируем последовательные одинаковые действия
            grouped_results = []
            current_group = [results[0]]
            
            for i in range(1, len(results)):
                if (results[i]['class_name'] == current_group[-1]['class_name'] and 
                    results[i]['confidence'] > 0.5):
                    current_group.append(results[i])
                else:
                    if len(current_group) >= 2:  # Группа должна быть минимум из 2 сегментов
                        grouped_results.append({
                            'class_name': current_group[0]['class_name'],
                            'start_time': current_group[0]['start_time'],
                            'end_time': current_group[-1]['end_time'],
                            'duration': current_group[-1]['end_time'] - current_group[0]['start_time'],
                            'avg_confidence': sum(r['confidence'] for r in current_group) / len(current_group)
                        })
                    current_group = [results[i]]
            
            # Добавляем последнюю группу
            if len(current_group) >= 2:
                grouped_results.append({
                    'class_name': current_group[0]['class_name'],
                    'start_time': current_group[0]['start_time'],
                    'end_time': current_group[-1]['end_time'],
                    'duration': current_group[-1]['end_time'] - current_group[0]['start_time'],
                    'avg_confidence': sum(r['confidence'] for r in current_group) / len(current_group)
                })
            
            # Сортируем по продолжительности
            grouped_results.sort(key=lambda x: x['duration'], reverse=True)
            
            # Выводим топ-5 самых продолжительных действий
            for i, group in enumerate(grouped_results[:5]):
                print(f"  {group['start_time']:.1f}-{group['end_time']:.1f}с "
                      f"({group['duration']:.1f}с): {group['class_name']} "
                      f"(уверенность: {group['avg_confidence']:.1%})")
    
    return results

In [ ]:
with open('../data/kinetics600-id2label.json', 'r', encoding='utf-8') as fp:
    class_names = json.load(fp)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = MoViNet(_C.MODEL.MoViNetA0, causal=True, pretrained=True)
model = model.to(device)

# Путь к видео
test_video_path = "../data/final_tracks_bot_tracker_resnet50/person_1.avi"

# Запускаем стриминг-анализ
results = predict_video_actions_streaming(
    model=model,
    video_path=test_video_path,
    class_names=class_names,
    device=device
)

# Сохраняем результаты в файл
import pandas as pd
if results:
    df = pd.DataFrame(results)
    df.to_csv('../artifacts/video_analysis_results.csv', index=False)
    print(f"\nРезультаты сохранены в video_analysis_results.csv")
    
    # Краткий отчет
    print(f"\nКРАТКИЙ ОТЧЕТ:")
    print(f"Всего сегментов: {len(results)}")
    print(f"Длительность видео: {results[-1]['end_time']:.1f} сек")
    
    # Наиболее частое действие
    most_common_action = df['class_name'].mode().iloc[0] if not df.empty else "N/A"
    print(f"Наиболее частое действие: {most_common_action}")

Using device: cpu
СТРИМИНГОВЫЙ АНАЛИЗ ВИДЕО
Видео: ../data/final_tracks/person_1.avi
Размер: 514x620, FPS: 20.00
Длительность: 372.0 сек, Кадров: 7440
Целевой размер: 172x172
Пропуск кадров: 1 из каждых 1.0 кадров

Параметры сегментации:
Длина сегмента: 16 кадров
Перекрытие: 50%
Шаг: 8 кадров

Начало стриминг-обработки...




































































































































































































































































































































































































































































































































































































































































































































































































































































































































































Обработка видео: 100%|██████████| 7440/7440 [02:48<00:00, 44.07кадр/s]


✓ Обработка завершена!
  Всего кадров: 7440
  Обработано кадров: 7440
  Создано сегментов: 929

РЕЗУЛЬТАТЫ АНАЛИЗА

Топ-5 действий в видео:
  playing lute: 616 сегментов (66.3%)
  playing violin: 188 сегментов (20.2%)
  playing guitar: 77 сегментов (8.3%)
  tapping guitar: 48 сегментов (5.2%)

Временная шкала основных действий:
  0.0-1.2с (1.2с): playing guitar (уверенность: 62.9%)

Результаты сохранены в video_analysis_results.csv

КРАТКИЙ ОТЧЕТ:
Всего сегментов: 929
Длительность видео: 372.0 сек
Наиболее частое действие: playing lute
